# Create new Ressource file

All Ressources 

In [1]:
import pandas as pd
from pathlib import Path

# Todo: Change
BASE = Path("/Users/simonimmler/PycharmProjects/Praktikum/bppso-groupwork") 
src = BASE /"resources" /"availabilities" /"availabilities_advanced.csv"
dst = (BASE /"resources" /"availabilities" /"availabilities_9to5.csv")

df = pd.read_csv(src)

df["StartTime"] = "09:00:00"
df["EndTime"]   = "17:00:00"
if "DurationMin" in df.columns:
    df["DurationMin"] = df["DurationMin"].fillna(0).astype(int)
    if "BreakStart" in df.columns:
        has_break = df["DurationMin"] > 0
        df.loc[has_break, "BreakStart"] = "13:00:00"
        df.loc[~has_break, "BreakStart"] = "00:00:00"

df.to_csv(dst, index=False)
print("Saved:", dst)
print(df.head())

Saved: /Users/simonimmler/PycharmProjects/Praktikum/bppso-groupwork/resources/availabilities/availabilities_9to5.csv
  Resource  DayId StartTime   EndTime BreakStart  DurationMin
0   User_1      0  09:00:00  17:00:00   00:00:00            0
1   User_1      1  09:00:00  17:00:00   00:00:00            0
2   User_1      2  09:00:00  17:00:00   00:00:00            0
3   User_1      3  09:00:00  17:00:00   00:00:00            0
4   User_1      4  09:00:00  17:00:00   00:00:00            0


Run and load metrics

In [2]:
%%capture
import pandas as pd
%run ./evaluation.ipynb

Compare the stats, (Ignore problems, they occur, because the metrics are loaded from the other notebook only at runtime

In [3]:
log_9to5 = pd.read_csv(BASE / "simulation_evaluation" /"results"/ "sim_output_9to5.csv",
                       parse_dates=["time:timestamp"])

avail_9to5_df = pd.read_csv(BASE / "resources" / "availabilities" / "availabilities_9to5.csv")
avail_9to5_df["StartTime"]= pd.to_datetime(avail_9to5_df["StartTime"], format="%H:%M:%S").dt.time
avail_9to5_df["EndTime"]= pd.to_datetime(avail_9to5_df["EndTime"], format="%H:%M:%S").dt.time
avail_9to5_df["DurationMin"] = avail_9to5_df["DurationMin"].fillna(0).astype(int)
log_9to5_completed = log_9to5[log_9to5["case:concept:name"].isin(
    log_9to5.groupby("case:concept:name").size()[lambda s: s > 2].index
)]

ct_9to5 = compute_cycle_times(log_9to5_completed)
work_9to5 = compute_working_seconds(log_9to5)
avail_9to5 = compute_available_seconds(avail_9to5_df)
occ_9to5 = compute_occupation(work_9to5, avail_9to5)
_, mad_9to5, wmad_9to5 = fairness_metrics(occ_9to5["occupation"], occ_9to5["available_seconds"])

compare_9to5 = pd.DataFrame({
    "Metric": [
        "# Completed Cases",
        "Avg Cycle Time (h)",
        "Avg Resource Occup (%)",
        "Fairness MAD (unwt)",
        "Fairness MAD (wtd)",
    ],
    "Advanced baseline": [
        len(advanced_ct),
        round(advanced_ct.mean(), 4),
        round(advanced_occ["occupation"].mean() * 100, 4),
        round(a_mad, 5),
        round(a_wmad, 5),
    ],
    "9-to-5": [
        len(ct_9to5),
        round(ct_9to5.mean(), 4),
        round(occ_9to5["occupation"].mean() * 100, 4),
        round(mad_9to5, 5),
        round(wmad_9to5, 5),
    ],
})

compare_9to5["Delta"] = compare_9to5["9-to-5"] - compare_9to5["Advanced baseline"]
print(compare_9to5.to_string(index=False))

                Metric  Advanced baseline     9-to-5     Delta
     # Completed Cases         2018.00000 2059.00000  41.00000
    Avg Cycle Time (h)            9.71470    3.94610  -5.76860
Avg Resource Occup (%)           21.82300   10.50150 -11.32150
   Fairness MAD (unwt)            0.26216    0.17035  -0.09181
    Fairness MAD (wtd)            0.26629    0.17670  -0.08959


Determine the maximum minutes after 5 p.m.

In [7]:
df = pd.read_csv(BASE/"simulation_evaluation/results/sim_output_9to5.csv")
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])
w = df[df["concept:name"].str.startswith("W_")].copy()
keys = ["case:concept:name", "concept:name", "org:resource"]
w["idx"] = w.groupby(keys + ["lifecycle:transition"]).cumcount()
starts = w[w["lifecycle:transition"] == "start"][keys + ["idx", "time:timestamp"]].rename(columns={"time:timestamp": "start_time"})
ends = w[w["lifecycle:transition"] == "complete"][keys + ["idx", "time:timestamp"]].rename(columns={"time:timestamp": "end_time"})

pairs = starts.merge(ends, on=keys + ["idx"], how="inner")

# after 5 pm
pairs["cutoff"] = pairs["end_time"].dt.normalize() + pd.Timedelta(hours=17)
pairs["minutes_after_17"] = (
    (pairs["end_time"] - pairs[["start_time", "cutoff"]].max(axis=1))
    .clip(lower=pd.Timedelta(0))
    .dt.total_seconds() / 60
)
print("Maximum minutes after 17:00:", pairs["minutes_after_17"].max())

Maximum minutes after 17:00: 33.45842531666667
